## Particle in a Box for Conjugated Dyes

A particle in a box (or particle on a line) is a model used to describe the energy and position of an electron confined in one dimension along a length $L$. The energy levels for the particle are given by

$$
E_n = \frac{n^2 h^2}{8mL^2},\quad n = 1,...,\infty
$$

For a conjugated dye, the particle is assumed to be an electron. 

In this notebook,

* All lengths are entered in units of Å. 
* Energy values are reported in J.


In [9]:
# Import necessary libraries

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.optimize import brentq
from IPython.display import display, Markdown

In [10]:
# Physical constants (SI)
h = 6.62607015e-34      # Planck constant, J*s
m_e = 9.1093837015e-31  # Electron mass, kg
c = 2.99792458e8        # Speed of light, m/s

ANGSTROM_TO_M = 1.0e-10


def pib_energies_joules(length_angstrom, n_levels):
    """Return quantum numbers and 1D PIB energies in Joules."""
    L_m = length_angstrom * ANGSTROM_TO_M
    n = np.arange(1, n_levels + 1)
    E_n = (n**2 * h**2) / (8.0 * m_e * L_m**2)
    return n, E_n


def max_quantum_number_for_energy(length_angstrom, energy_max_j):
    """Largest integer n with E_n <= energy_max_j for a given box length."""
    if energy_max_j <= 0:
        return 0

    L_m = length_angstrom * ANGSTROM_TO_M
    n_max = np.sqrt((8.0 * m_e * (L_m**2) * energy_max_j) / (h**2))
    return int(np.floor(n_max))



In [ ]:
# User controls for the energy-level display
length_widget = widgets.FloatSlider(
    value=12.0,
    min=1.0,
    max=30.0,
    step=0.01,
    description="L (Angstrom)",
    continuous_update=True,
)

energy_max_widget = widgets.FloatLogSlider(
    value=2.0e-18,
    base=10,
    min=-20,
    max=-16,
    step=0.01,
    description="E max (J)",
    readout_format=".2e",
    continuous_update=True,
)

selected_level_widget = widgets.IntSlider(
    value=1,
    min=1,
    max=1,
    step=1,
    description="State n",
    continuous_update=False,
)

show_state_widget = widgets.Checkbox(
    value=False,
    description="Show selected state",
)

view_widget = widgets.ToggleButtons(
    options=[("Wavefunction psi", "psi"), ("Probability |psi|^2", "prob")],
    value="psi",
    description="View",
)



def plot_energy_levels(length_angstrom, energy_max_j):
    n_levels = max_quantum_number_for_energy(length_angstrom, energy_max_j)
    if n_levels < 1:
        print("Increase E max: no energy levels fit in the current y-axis range.")
        return

    n, E_n = pib_energies_joules(length_angstrom, n_levels)

    selected_level_widget.max = n_levels

    fig, ax = plt.subplots(figsize=(7, 6))

    # Box is centered at L/2.
    x_left, x_right = 0, length_angstrom
    x_fixed_half_range = (30 - length_angstrom) / 2

    ax.plot([x_left, x_left], [0, energy_max_j], color="black", linewidth=2)
    ax.plot([x_right, x_right], [0, energy_max_j], color="black", linewidth=2)

    for i, energy in enumerate(E_n):
        ax.hlines(energy, x_left, x_right, colors="tab:blue", linewidth=2)
        ax.text(
            min(x_right + 0.35, length_angstrom + 1),
            energy,
            f"n={n[i]}",
            va="center",
            fontsize=9,
        )

    ax.set_xlim(-x_fixed_half_range, length_angstrom + x_fixed_half_range)
    ax.set_ylim(0.0, energy_max_j)
    ax.set_xlabel("Position (Angstrom)")
    ax.set_ylabel("Energy (J)")
    ax.set_title(
        f"1D Particle-in-a-Box Energy Levels (L = {length_angstrom:.1f} Angstrom, E max = {energy_max_j:.2e} J)"
    )
    plt.show()


def infinite_well_wavefunction(length_angstrom, n_state, x_angstrom):
    """Return normalized finite-well bound-state wavefunction values psi(x)."""


   # x_m = np.asarray(x_angstrom) * ANGSTROM_TO_M
    

    psi = np.zeros_like(x_angstrom, dtype=float)

    psi = np.sin(n_state * np.pi * x_angstrom / length_angstrom)

    norm = np.trapezoid(psi**2, x_angstrom)
    if norm > 0:
        psi = psi / np.sqrt(norm)

    return psi


def plot_selected_state(length_angstrom, energy_max_j, selected_n, show_state, view_mode):
    n_levels = max_quantum_number_for_energy(length_angstrom, energy_max_j)
    x_wave_angstrom = np.linspace(0, length_angstrom, 1500)
   # states = finite_well_bound_states(length_angstrom, well_depth_j)
   # n_bound = len(states["E_n"])

    if n_levels < 1:
        print("No bound states available for wavefunction display.")
        return

    if selected_n > n_levels:
        print(f"Selected state n={selected_n} is not bound at current depth.")
        return

    if not show_state:
        print("Click an energy level (or check 'Show selected state') to display a state profile.")
        return

    psi = infinite_well_wavefunction(length_angstrom, selected_n, x_wave_angstrom)
    y = psi if view_mode == "psi" else psi**2

    x_fixed_half_range = (30 - length_angstrom) / 2

    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(x_wave_angstrom, y, color="tab:purple", linewidth=2)
    x_left, x_right = 0, length_angstrom
    ax.axvline(x_left, color="black", linestyle="--", linewidth=1)
    ax.axvline(x_right, color="black", linestyle="--", linewidth=1)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_xlim(-x_fixed_half_range, length_angstrom + x_fixed_half_range)
    ax.set_xlabel("Position (Angstrom)")
    if view_mode == "psi":
        ax.set_ylabel("ψ(x)")
        ax.set_title(f"Particle in a Box Wavefunction for n={selected_n}")
    else:
        ax.set_ylabel("|ψ(x)|^2")
        ax.set_title(f"Particle in a Box Probability Distribution for n={selected_n}")
    plt.show()


display(Markdown("### Energy Level Display"))
display(widgets.HBox([length_widget, energy_max_widget]))
display(Markdown("Select an energy level for wavefunction display."))
display(widgets.HBox([selected_level_widget, show_state_widget, view_widget]))
energydisplay = widgets.interactive_output(
    plot_energy_levels,
    {"length_angstrom": length_widget, "energy_max_j": energy_max_widget},
)

statedisplay = widgets.interactive_output(
    plot_selected_state,
    {
        "length_angstrom": length_widget,
        "energy_max_j": energy_max_widget,
        "selected_n": selected_level_widget,
        "show_state": show_state_widget,
        "view_mode": view_widget,
    }
)

display(energydisplay)
display(statedisplay)

### Energy Level Display

Select an energy level for wavefunction display.

Output()

Output()

In [12]:
# Display energy level table
def show_energy_table(length_angstrom, energy_max_j):
    n_levels = max_quantum_number_for_energy(length_angstrom, energy_max_j)
    if n_levels < 1:
        display(Markdown("### Optional Table View"))
        display(Markdown("No levels fall at or below the selected maximum energy."))
        return

    n, E_n = pib_energies_joules(length_angstrom, n_levels)

    lines = ["| n | Energy (J) |", "|---:|---:|"]
    lines.extend([f"| {ni} | {energy:.3e} |" for ni, energy in zip(n, E_n)])

    display(Markdown("### Optional Table View"))
    display(Markdown("\n".join(lines)))


table_out = widgets.interactive_output(
    show_energy_table,
    {"length_angstrom": length_widget, "energy_max_j": energy_max_widget},
)
display(table_out)

Output()

### Predict the value of $\lambda_\text{max}$

* Count the number of π-electrons in the dye
* Identify the HOMO and LUMO and the corresponding energies from the table
* Use the code cell to calculate $\Delta E$ (in J) and $\lambda_\text{max}$ (in nm)

In [17]:
# Using 4,4'-dicarbocyanine with 14 π-electrons and a box length of 19.2 Å

n_HOMO = 7
n_LUMO = 8
E_HOMO = 8.008e-19
E_LUMO = 1.046e-18

delta_E = E_LUMO - E_HOMO

lambda_max = (h * c) / delta_E * 1.e9

print("λ_max = ", f"{lambda_max:.2f} nm")

λ_max =  810.13 nm


In [18]:

def homo_lumo_info(length_angstrom, n_levels, n_pi_electrons):
    """Compute HOMO/LUMO indices, energies, gap, and wavelength in nm."""
    if n_pi_electrons % 2 != 0:
        raise ValueError("pi-electron count must be even for this model")

    homo_n = n_pi_electrons // 2
    lumo_n = homo_n + 1

    if n_levels < lumo_n:
        raise ValueError(
            f"Increase displayed levels to at least {lumo_n} to include the LUMO"
        )

    n, E_n = pib_energies_joules(length_angstrom, n_levels)
    E_homo = E_n[homo_n - 1]
    E_lumo = E_n[lumo_n - 1]
    delta_E = E_lumo - E_homo

    wavelength_m = (h * c) / delta_E
    wavelength_nm = wavelength_m * 1.0e9

    return {
        "homo_n": homo_n,
        "lumo_n": lumo_n,
        "E_homo_J": E_homo,
        "E_lumo_J": E_lumo,
        "delta_E_J": delta_E,
        "wavelength_nm": wavelength_nm,
        "n": n,
        "E_n": E_n,
    }

In [15]:
pi_electrons_widget = widgets.IntSlider(
    value=10,
    min=2,
    max=60,
    step=2,
    description="pi e-",
    continuous_update=False,
)


def show_homo_lumo(length_angstrom, energy_max_j, n_pi_electrons):
    n_levels = max_quantum_number_for_energy(length_angstrom, energy_max_j)
    if n_levels < 1:
        print("Input issue: increase E max so at least one energy level is visible.")
        return

    try:
        result = homo_lumo_info(length_angstrom, n_levels, n_pi_electrons)
    except ValueError as exc:
        print(f"Input issue: {exc}")
        return

    print(f"Length, L: {length_angstrom:.2f} Angstrom")
    print(f"Maximum displayed energy: {energy_max_j:.2e} J")
    print(f"Displayed levels from E max: {n_levels}")
    print(f"pi-electrons: {n_pi_electrons}")
    print()
    print(f"HOMO level: n = {result['homo_n']}")
    print(f"LUMO level: n = {result['lumo_n']}")
    print(f"E_HOMO: {result['E_homo_J']:.6e} J")
    print(f"E_LUMO: {result['E_lumo_J']:.6e} J")
    print(f"Delta E (HOMO->LUMO): {result['delta_E_J']:.6e} J")
    print(f"Predicted transition wavelength: {result['wavelength_nm']:.2f} nm")


display(Markdown("### HOMO-LUMO Information"))
display(widgets.HBox([length_widget, energy_max_widget, pi_electrons_widget]))
widgets.interactive_output(
    show_homo_lumo,
    {
        "length_angstrom": length_widget,
        "energy_max_j": energy_max_widget,
        "n_pi_electrons": pi_electrons_widget,
    },
)

### HOMO-LUMO Information

Output()

In [16]:
# Quick verification examples
show_homo_lumo(length_angstrom=12.0, energy_max_j=2.0e-18, n_pi_electrons=10)

energy_max_verify = 5.0e-18
n_small = max_quantum_number_for_energy(8.0, energy_max_verify)
n_large = max_quantum_number_for_energy(16.0, energy_max_verify)
r_small = homo_lumo_info(length_angstrom=8.0, n_levels=n_small, n_pi_electrons=10)
r_large = homo_lumo_info(length_angstrom=16.0, n_levels=n_large, n_pi_electrons=10)

print()
print("Trend check (fixed electrons):")
print(
    f"L=8.0 Angstrom -> Delta E={r_small['delta_E_J']:.3e} J, lambda={r_small['wavelength_nm']:.1f} nm"
)
print(
    f"L=16.0 Angstrom -> Delta E={r_large['delta_E_J']:.3e} J, lambda={r_large['wavelength_nm']:.1f} nm"
)
if r_large['delta_E_J'] < r_small['delta_E_J'] and r_large['wavelength_nm'] > r_small['wavelength_nm']:
    print("Sanity check passed: larger box length gives smaller gap and longer wavelength.")
else:
    print("Sanity check failed: re-check formulas/units.")

Length, L: 12.00 Angstrom
Maximum displayed energy: 2.00e-18 J
Displayed levels from E max: 6
pi-electrons: 10

HOMO level: n = 5
LUMO level: n = 6
E_HOMO: 1.045949e-18 J
E_LUMO: 1.506167e-18 J
Delta E (HOMO->LUMO): 4.602176e-19 J
Predicted transition wavelength: 431.63 nm

Trend check (fixed electrons):
L=8.0 Angstrom -> Delta E=1.035e-18 J, lambda=191.8 nm
L=16.0 Angstrom -> Delta E=2.589e-19 J, lambda=767.3 nm
Sanity check passed: larger box length gives smaller gap and longer wavelength.
